# Object Analysis -> Target Discovery

This notebook runs the two public workflows end to end:

1. `object_analysis`: Cellpose object detection, classical features, optional DINOv2 embeddings, and the object table.
2. `target_discovery`: embedding clustering plus target selection.

The example uses `skimage.data.human_mitosis()`, which is provided through pooch by scikit-image. Outputs are plotted inline: detected objects on the source image, UMAP clusters, and selected targets on the image.

Required conda environments:

- `SMART--object_analysis--vision`
- `SMART--object_analysis--classical`
- `SMART--target_discovery--main`
- `SMART--target_discovery--cluster`

The first run may download the human mitosis image through pooch and DINOv2 weights through Torch Hub.

## Setup

In [ ]:
from __future__ import annotations

import json
import os
import sys
import time
from pathlib import Path

import numpy as np
import tifffile
from IPython.display import HTML, SVG, display
from skimage.data import human_mitosis

try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in (current, *current.parents):
        if (candidate / "engine").exists() and (candidate / "workflows").exists():
            return candidate
    raise RuntimeError("Could not find smart-analysis repo root.")


ROOT = find_repo_root(Path.cwd())
WORKFLOWS = ROOT / "workflows"
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(WORKFLOWS))

from engine import Engine  # noqa: E402
from _contracts import save_overview  # noqa: E402

RUN_DIR = ROOT / "output" / "notebooks" / "object_analysis_target_discovery"
OBJECT_DIR = RUN_DIR / "object_analysis"
DISCOVERY_DIR = RUN_DIR / "target_discovery"
IMAGE_PATH = RUN_DIR / "human_mitosis.ome.tiff"
OVERVIEW_PATH = RUN_DIR / "overview.json"

RUN_DIR.mkdir(parents=True, exist_ok=True)
OBJECT_DIR.mkdir(parents=True, exist_ok=True)
DISCOVERY_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repo: {ROOT}")
print(f"Run folder: {RUN_DIR}")

If the notebook kernel cannot find conda, set `CONDA_EXE` before running workflow cells. Keep this generic in committed examples; do not hard-code a machine-specific path here.

In [ ]:
# Example, only if conda is not on PATH in this notebook kernel:
# os.environ["CONDA_EXE"] = r"path/to/conda.exe"

print("CONDA_EXE:", os.environ.get("CONDA_EXE", "<not set>"))

## Load the example image

In [ ]:
image = human_mitosis()
tifffile.imwrite(IMAGE_PATH, image, photometric="minisblack")

print("Image:", IMAGE_PATH)
print("Shape:", image.shape, "dtype:", image.dtype)

if plt is not None:
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.imshow(image, cmap="gray")
    ax.set_title("human_mitosis input image")
    ax.set_axis_off()
    plt.show()
else:
    display(HTML("<p>matplotlib is unavailable; image plotting skipped.</p>"))

## Helpers

In [ ]:
def run_engine_workflow(name: str, yaml_path: Path, payload: dict, timeout_s: int = 900) -> dict:
    with Engine() as engine:
        engine.register(name, str(yaml_path))
        engine.submit(name, payload)
        deadline = time.monotonic() + timeout_s
        while time.monotonic() < deadline:
            results = engine.results(name)
            if results:
                return results[0]
            status = engine.status(name)
            if status["failed"]:
                failure = status["failures"][0]
                raise RuntimeError(f"{failure['step']}: {failure['error']}")
            time.sleep(0.2)
    raise TimeoutError(f"Timed out waiting for workflow {name!r}.")


def rows_from_properties(tile: dict) -> list[dict]:
    objects = tile["objects"]
    props = objects["properties"]
    rows = []
    for idx in range(objects["n_objects"]):
        rows.append({key: values[idx] for key, values in props.items()})
    return rows


def html_table(rows: list[dict], columns: list[str], max_rows: int = 12):
    if not rows:
        return HTML("<p>No rows.</p>")
    head = "".join(f"<th>{col}</th>" for col in columns)
    body = []
    for row in rows[:max_rows]:
        cells = "".join(f"<td>{row.get(col, '')}</td>" for col in columns)
        body.append(f"<tr>{cells}</tr>")
    suffix = "" if len(rows) <= max_rows else f"<p>Showing {max_rows} of {len(rows)} rows.</p>"
    return HTML(f"<table><thead><tr>{head}</tr></thead><tbody>{''.join(body)}</tbody></table>{suffix}")

## Run workflow 1: object analysis

This runs the deep-feature pipeline so the downstream clustering workflow has embeddings. The public object table keeps absolute `stage_x_um` / `stage_y_um` coordinates derived from the input tile geometry.

In [ ]:
object_yaml = ROOT / "workflows" / "object_analysis" / "pipelines" / "object_analysis_deep.yaml"

object_payload = {
    "image_path": str(IMAGE_PATH),
    "tile_id": ["R0", 0, 0],
    "tile_stage_xy_um": [10000.0, 15000.0],
    "tile_zwide_um": 0.0,
    "source_pixel_size_um": [0.65, 0.65],
    "source_image_size_px": [int(image.shape[1]), int(image.shape[0])],
    "image_to_stage": [[0.0, -1.0], [1.0, 0.0]],
    "channels": None,
    "gpu": True,
    "backend": "dinov2",
    "model_name": "dinov2_vitb14",
    "input_size_px": 224,
    "batch_size": 1,
    "output_dir": str(OBJECT_DIR),
}

object_result = run_engine_workflow("object_analysis_notebook", object_yaml, object_payload, timeout_s=1200)
tile = object_result["object_analysis"]
save_overview(OVERVIEW_PATH, {"tiles": [tile]})

objects = tile["objects"]
props = objects["properties"]
embeddings = objects.get("embeddings")
print("Objects:", objects["n_objects"])
print("Embeddings:", "yes" if embeddings else "no")
if embeddings:
    print("Embedding matrix:", len(embeddings["vectors"]), "x", len(embeddings["vectors"][0]) if embeddings["vectors"] else 0)
print("Overview:", OVERVIEW_PATH)

## Plot object-analysis output

In [ ]:
object_rows = rows_from_properties(tile)
display(html_table(
    object_rows,
    ["object_id", "label", "area", "intensity_mean", "stage_x_um", "stage_y_um"],
    max_rows=10,
))

if plt is not None:
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(image, cmap="gray")
    scatter = ax.scatter(
        props["centroid_col_px"],
        props["centroid_row_px"],
        c=props["area"],
        cmap="viridis",
        s=42,
        edgecolors="white",
        linewidths=0.6,
    )
    ax.set_title("Detected objects colored by area")
    ax.set_axis_off()
    fig.colorbar(scatter, ax=ax, fraction=0.046, pad=0.04, label="area")
    plt.show()
else:
    display(HTML("<p>matplotlib is unavailable; object plot skipped.</p>"))

## Run workflow 2: target discovery with clustering

This runs `cluster_objects -> select_targets`. Clustering is performed in embedding space using cosine kNN, Leiden, and UMAP. The output includes both a table and a scatterplot artifact.

In [ ]:
target_yaml = ROOT / "workflows" / "target_discovery" / "pipelines" / "target_discovery_cluster.yaml"
n_objects = tile["objects"]["n_objects"]
if n_objects < 1:
    raise RuntimeError("No objects were detected; adjust Cellpose channels before running target discovery.")
if not tile["objects"].get("embeddings"):
    raise RuntimeError("Clustering requires embeddings; run object_analysis_deep.yaml before target discovery.")

target_payload = {
    "tiles": [tile],
    "output_dir": str(DISCOVERY_DIR),
    "n_neighbors": max(2, min(15, n_objects - 1)),
    "leiden_resolution": 1.0,
    "random_state": 0,
    "umap_min_dist": 0.1,
    "feature": "area",
    "direction": "high",
    "n_per_tile": 8,
    "border_margin_px": 0,
}

target_result = run_engine_workflow("target_discovery_notebook", target_yaml, target_payload, timeout_s=600)
discovery = target_result["target_discovery"]
clusters = discovery["clusters"]
targets = discovery["targets"]

print("Clustered objects:", clusters["n_objects"])
print("Clusters:", clusters["n_clusters"])
print("Targets:", len(targets))
print(json.dumps(clusters.get("artifacts", {}), indent=2))

## Plot clustering output

In [ ]:
cluster_rows = clusters["table"]
display(html_table(
    cluster_rows,
    ["object_id", "cluster_id", "umap_x", "umap_y", "stage_x_um", "stage_y_um", "area"],
    max_rows=12,
))

svg_path = clusters.get("artifacts", {}).get("cluster_plot_svg")
if svg_path and Path(svg_path).exists():
    display(SVG(filename=svg_path))
elif plt is not None:
    fig, ax = plt.subplots(figsize=(7, 5))
    xs = [row["umap_x"] for row in cluster_rows]
    ys = [row["umap_y"] for row in cluster_rows]
    cs = [row["cluster_id"] for row in cluster_rows]
    ax.scatter(xs, ys, c=cs, cmap="tab10", s=40, edgecolors="black", linewidths=0.4)
    ax.set_title("UMAP clusters")
    ax.set_xlabel("umap_x")
    ax.set_ylabel("umap_y")
    plt.show()
else:
    display(HTML("<p>No cluster plot artifact found.</p>"))

## Plot selected targets on the source image

In [ ]:
display(html_table(
    targets,
    ["target_id", "object_label", "score", "source_feature", "stage_x_um", "stage_y_um"],
    max_rows=12,
))

target_labels = {target["object_label"] for target in targets}
target_indices = [idx for idx, label in enumerate(props["label"]) if label in target_labels]

if plt is not None:
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(image, cmap="gray")
    ax.scatter(
        props["centroid_col_px"],
        props["centroid_row_px"],
        s=28,
        facecolors="none",
        edgecolors="deepskyblue",
        linewidths=0.8,
        label="all objects",
    )
    ax.scatter(
        [props["centroid_col_px"][idx] for idx in target_indices],
        [props["centroid_row_px"][idx] for idx in target_indices],
        s=90,
        marker="x",
        c="red",
        linewidths=2.0,
        label="selected targets",
    )
    ax.set_title("Selected targets on the source image")
    ax.set_axis_off()
    ax.legend(loc="lower right")
    plt.show()
else:
    display(HTML("<p>matplotlib is unavailable; target overlay skipped.</p>"))

## Output files

In [ ]:
artifact_rows = [
    {"artifact": "overview", "path": str(OVERVIEW_PATH)},
    {"artifact": "object_analysis", "path": str(OBJECT_DIR)},
    {"artifact": "target_discovery", "path": str(DISCOVERY_DIR)},
]
for key, path in clusters.get("artifacts", {}).items():
    artifact_rows.append({"artifact": key, "path": path})

display(html_table(artifact_rows, ["artifact", "path"], max_rows=20))